Recomendation Models

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer

In [4]:
df = pd.read_csv('/content/drive/MyDrive/New Project/scotch_review2020.csv')

In [5]:
df.head()

,id,name,category,review.point,price,currency,description.1.2247.
0,1,"Black Bowmore 42 year old 1964 vintage, 40.5%",Single Malt Scotch,97,4500,$,What impresses me most is how this whisky evol...
1,2,"Bowmore 46 year old (distilled 1964), 42.9%",Single Malt Scotch,97,13500,$,There have been some legendary Bowmores from t...
2,3,"Johnnie Walker Blue Label, 40%",Blended Scotch Whisky,97,225,$,"Magnificently powerful and intense. Caramels, ..."
3,4,"Glenlivet Cellar Collection 1969 vintage, 50.8%",Single Malt Scotch,96,750,$,It’s great that Glenlivet releases whiskies un...
4,5,The Macallan 29 year old 1976 Vintage (Cask #1...,Single Malt Scotch,96,"1,500",$,Classic sherry cask-aged Macallan. Antique amb...


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2247 entries, 0 to 2246
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   id                   2247 non-null   int64 
 1   name                 2247 non-null   object
 2   category             2247 non-null   object
 3   review.point         2247 non-null   int64 
 4   price                2247 non-null   object
 5   currency             2247 non-null   object
 6   description.1.2247.  2208 non-null   object
dtypes: int64(2), object(5)
memory usage: 123.0+ KB


In [7]:
df['category'].value_counts()

,count
category,
Single Malt Scotch,1835
Blended Scotch Whisky,247
Blended Malt Scotch Whisky,165


In [8]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import hstack

print("Starting setup for the recommendation model...")

# Rename columns for easier access
df.rename(columns={'description.1.2247.': 'description', 'review.point': 'review_point'}, inplace=True)

# Fill missing descriptions with empty strings
df['description'] = df['description'].fillna('')

# Clean and convert 'price' to numeric, handling missing values
df['price'] = df['price'].astype(str).str.replace(',', '', regex=False)
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df['price'] = df['price'].fillna(df['price'].median())
print("Data cleaning for 'description' and 'price' complete.")

# 1. TF-IDF Vectorization for 'description'
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = tfidf_vectorizer.fit_transform(df['description'])
print("TF-IDF matrix shape:", tfidf_matrix.shape)

# 2. Scale 'price' using MinMaxScaler
scaler = MinMaxScaler()
df['price_scaled'] = scaler.fit_transform(df[['price']])
print("Price column scaled.")

# 3. Combine TF-IDF matrix and scaled 'price'
# Use hstack for sparse matrices to combine with dense price_scaled feature
combined_features = hstack([tfidf_matrix, df[['price_scaled']]])
print("Combined features shape:", combined_features.shape)

# 4. Cosine Similarity on combined features
cosine_sim = cosine_similarity(combined_features)
print("Cosine similarity matrix shape:", cosine_sim.shape)

# 5. Define the get_recommendations function
def get_recommendations(whisky_name, cosine_sim_matrix, df_data, num_recommendations=5):
    # Find the index of the whisky that matches the name
    idx = df_data[df_data['name'].str.contains(whisky_name, case=False, na=False)].index

    if len(idx) == 0:
        print(f"Whisky '{whisky_name}' not found in the dataset.")
        return pd.DataFrame()

    whisky_idx = idx[0]

    # Get pairwise similarity scores of all whiskies with that whisky
    sim_scores = list(enumerate(cosine_sim_matrix[whisky_idx]))

    # Sort the whiskies based on the similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the scores of the most similar whiskies (excluding itself)
    sim_scores = sim_scores[1:num_recommendations+1]

    # Get the whisky indices
    whisky_indices = [i[0] for i in sim_scores]

    # Return the top N most similar whiskies with specific columns
    return df_data.iloc[whisky_indices][['name', 'category', 'price', 'currency', 'description']]

print("Recommendation model setup complete. You can now use the 'get_recommendations' function.")

Starting setup for the recommendation model...
Data cleaning for 'description' and 'price' complete.
TF-IDF matrix shape: (2247, 5000)
Price column scaled.
Combined features shape: (2247, 5001)
Cosine similarity matrix shape: (2247, 2247)
Recommendation model setup complete. You can now use the 'get_recommendations' function.


### Get Whisky Recommendations

Now that the recommendation model is set up, you can get recommendations by entering a whisky name below.

In [33]:

user_whisky = input("Enter data: ").strip()

print(f"\nSearching for details of '{user_whisky}'...")

# First, try to find an exact match (case-insensitive)
exact_match = df[df['name'].str.lower() == user_whisky.lower()]

if not exact_match.empty:
    if len(exact_match) == 1:
        print(f"Here are the details for '{user_whisky}':")
        display(exact_match)
    else:
        # This case is less likely for whisky names but handles multiple exact matches if they exist
        print(f"Multiple exact matches found for '{user_whisky}'. Showing all details:")
        display(exact_match)
else:
    # If no exact match, find all whiskies containing the input string
    partial_matches = df[df['name'].str.contains(user_whisky, case=False, na=False)]

    if not partial_matches.empty:
        # Sort by review_point in descending order to get the highest-rated one
        # and display only the top result
        top_match = partial_matches.sort_values(by='review_point', ascending=False).iloc[[0]]
        print(f"No exact match found for '{user_whisky}'. Showing details for the top-rated similar whisky:")
        display(top_match)
    else:
        print(f"Could not find any whisky matching '{user_whisky}' in the dataset. Please try a different name.")

Enter data: Johnnie Walker black Label

Searching for details of 'Johnnie Walker black Label'...
No exact match found for 'Johnnie Walker black Label'. Showing details for the top-rated similar whisky:


,id,name,category,review_point,price,currency,description,price_scaled
1053,1054,"Johnnie Walker Black Label Highlands Origin, 42%",Blended Malt Scotch Whisky,89,46.0,$,This expression is described by the distillers...,0.000229
